# 04 · Selektor z podglądem a losowy wybór (MoNuSeg)

**Pytanie.** Czy wybieranie obrazów, na których student faktycznie wypada najgorzej, uczy go szybciej niż losowanie?

**Selektor z podglądem** (`oracle_peek`) po każdej rundzie segmentuje studentem całą pulę nieoznaczoną, porównuje maski
z maskami wyroczni i bierze 2 obrazy o najniższym PQ. Czyta etykiety puli, więc **nie jest metodą**, tylko górną granicą
dla selektorów typu „wybieraj trudne” (niepewność, system rozmyty).

**Porównanie sparowane:** dla seeda *k* oba przebiegi startują z tego samego `student_init.pth` i tych samych 2 obrazów,
z tymi samymi hiperparametrami treningu. Runda 2 jest więc tym samym eksperymentem powtórzonym dwa razy i pokazuje
poziom szumu między uruchomieniami.

Miara główna: PQ maski studenta względem maski wyroczni na walidacji (`val_oracle_pq`). Zbiór testowy nie jest używany.

Dane: `outputs/al_random/full` i `outputs/al_oracle_peek/full` (WCSS, `slurm/monuseg_random_al.sbatch`).

In [ ]:
SEEDS = [0, 1, 2, 3, 4, 5]
ROUNDS = [2, 4, 6, 8, 10, 12]  # rundy wspólne dla obu przebiegów
METRIC = "val_oracle_pq"

In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from histopathology_pnw.config import Settings
from histopathology_pnw.datasets import monuseg

OUT = Settings().outputs_dir


def load(selector: str) -> pd.DataFrame:
    frames = []
    for seed in SEEDS:
        df = pd.read_csv(OUT / f"al_{selector}" / "full" / f"seed{seed}" / "results.csv")
        frames.append(df[df["n_labeled"].isin(ROUNDS)].assign(seed=seed))
    return pd.concat(frames, ignore_index=True)


rand, peek = load("random"), load("oracle_peek")
missing = {s: sorted(set(ROUNDS) - set(peek.loc[peek["seed"] == s, "n_labeled"])) for s in SEEDS}
print("brakujące rundy przebiegu z podglądem:", {s: m for s, m in missing.items() if m} or "brak")


def wide(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    return df.pivot(index="seed", columns="n_labeled", values=metric)


r, p = wide(rand, METRIC), wide(peek, METRIC)
diff = p - r

## Poziom szumu: runda 2

W rundzie 2 oba przebiegi trenują ten sam model na tych samych 2 obrazach. Różnica to niedeterminizm treningu na GPU,
czyli najmniejsza różnica, jaką w ogóle da się przypisać selektorowi.

In [ ]:
noise = pd.DataFrame({"losowy": r[2], "podgląd": p[2], "różnica": diff[2]})
print(
    f"runda 2: średnia |różnica| = {diff[2].abs().mean():.4f}, największa = {diff[2].abs().max():.4f}"
)
noise.round(4)

## Krzywe uczenia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, w, color in (("losowy", r, "tab:gray"), ("z podglądem", p, "tab:red")):
    for seed in w.index:
        axes[0].plot(w.columns, w.loc[seed], color=color, alpha=0.25, lw=1)
    axes[0].errorbar(
        w.columns,
        w.mean(),
        yerr=w.std(ddof=1),
        color=color,
        lw=2,
        marker="o",
        capsize=3,
        label=name,
    )
axes[0].set(
    xlabel="liczba oznaczonych obrazów",
    ylabel="PQ studenta względem wyroczni (walidacja)",
    title="Średnia ± odch. std. po seedach; cienkie linie = seedy",
    xticks=ROUNDS,
)
axes[0].legend()

for seed in diff.index:
    axes[1].plot(diff.columns, diff.loc[seed], marker=".", alpha=0.6, label=f"seed {seed}")
axes[1].plot(diff.columns, diff.mean(), color="k", lw=2.5, marker="o", label="średnia")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].axhspan(
    -diff[2].abs().max(), diff[2].abs().max(), color="gray", alpha=0.15, label="szum (runda 2)"
)
axes[1].set(
    xlabel="liczba oznaczonych obrazów",
    ylabel="podgląd minus losowy (PQ)",
    title="Różnica na seed (porównanie sparowane)",
    xticks=ROUNDS,
)
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
fig_dir = OUT / "al_oracle_peek" / "analysis"
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "peek_vs_random.png", dpi=130, bbox_inches="tight")

## Różnice w rundach i ALC

Dla każdej rundy: średnia różnica po seedach, liczba seedów, w których podgląd był lepszy, test t dla par
(i test Wilcoxona; przy 6 seedach najmniejsze możliwe p dwustronne to 0,031).

ALC(2–12) = pole pod krzywą metodą trapezów, podzielone przez długość przedziału (czyli średnie PQ w oknie 2–12).

In [ ]:
def paired(d: pd.Series) -> dict:
    d = d.dropna()
    t = stats.ttest_1samp(d, 0.0)
    try:
        w = stats.wilcoxon(d).pvalue
    except ValueError:
        w = np.nan
    return {
        "średnia różnica": float(d.mean()),
        "odch. std.": float(d.std(ddof=1)),
        "seedy z podglądem lepszym": f"{int((d > 0).sum())}/{len(d)}",
        "p (t)": float(t.pvalue),
        "p (Wilcoxon)": float(w),
    }


def alc(w: pd.DataFrame) -> pd.Series:
    x = np.array(w.columns, dtype=float)
    return w.apply(lambda row: np.trapezoid(row.to_numpy(), x) / (x[-1] - x[0]), axis=1)


per_round = pd.DataFrame({n: paired(diff[n]) for n in ROUNDS}).T
per_round.insert(0, "losowy", r.mean())
per_round.insert(1, "podgląd", p.mean())
per_round.index.name = "n_labeled"
alc_tab = pd.DataFrame({"losowy": alc(r), "podgląd": alc(p)})
alc_tab["różnica"] = alc_tab["podgląd"] - alc_tab["losowy"]
print(
    "ALC(2-12):",
    {
        k: (round(v, 4) if isinstance(v, float) else v)
        for k, v in paired(alc_tab["różnica"]).items()
    },
)
print(
    f"ALC losowy {alc_tab['losowy'].mean():.4f} ± {alc_tab['losowy'].std(ddof=1):.4f}, "
    f"podgląd {alc_tab['podgląd'].mean():.4f} ± {alc_tab['podgląd'].std(ddof=1):.4f}"
)
per_round.to_csv(fig_dir / "per_round.csv")
alc_tab.to_csv(fig_dir / "alc.csv")
per_round

In [ ]:
alc_tab.round(4)

## To samo względem ground truth

Kontrola: czy wniosek jest taki sam, gdy studenta oceniamy względem GT MoNuSeg (`val_pq`), a nie względem wyroczni.

In [ ]:
r_gt, p_gt = wide(rand, "val_pq"), wide(peek, "val_pq")
d_gt = p_gt - r_gt
gt_tab = pd.DataFrame({n: paired(d_gt[n]) for n in ROUNDS}).T
gt_tab.insert(0, "losowy", r_gt.mean())
gt_tab.insert(1, "podgląd", p_gt.mean())
a_gt = alc(p_gt) - alc(r_gt)
print(
    "ALC(2-12) względem GT:",
    {k: (round(v, 4) if isinstance(v, float) else v) for k, v in paired(a_gt).items()},
)
gt_tab

## Co wybierał podgląd

`selection.csv` z każdej rundy: PQ studenta względem wyroczni dla każdego obrazu puli i to, które zostały wybrane.

In [ ]:
POOL = monuseg.load_train_val_split()["train"]
# narząd jest znany tylko dla części obrazów (PDF dołączony do MoNuSeg)
organs = monuseg.organs().set_index("image_id")["organ"].reindex(POOL).fillna("nieznany")
sel = pd.concat(
    [
        pd.read_csv(
            OUT / "al_oracle_peek" / "full" / f"seed{s}" / f"round_{n:02d}" / "selection.csv"
        ).assign(seed=s, n_labeled=n)
        for s in SEEDS
        for n in ROUNDS
        if (
            OUT / "al_oracle_peek" / "full" / f"seed{s}" / f"round_{n:02d}" / "selection.csv"
        ).is_file()
    ],
    ignore_index=True,
)
sel["organ"] = sel["image_id"].map(organs)
sel.to_csv(fig_dir / "selection_all.csv", index=False)

pool_pq = sel.groupby("n_labeled").apply(
    lambda g: pd.Series(
        {
            "PQ puli: średnia": g["pq_oracle"].mean(),
            "PQ puli: min": g["pq_oracle"].min(),
            "PQ puli: odch. std.": g["pq_oracle"].std(ddof=1),
            "PQ wybranych": g.loc[g["selected"], "pq_oracle"].mean(),
        }
    ),
    include_groups=False,
)
pool_pq.round(3)

In [ ]:
chosen = sel[sel["selected"]]
pool_share = organs.value_counts(normalize=True)
picked_share = chosen["organ"].value_counts(normalize=True)
organ_tab = (
    pd.DataFrame({"udział w puli": pool_share, "udział wśród wybranych": picked_share})
    .fillna(0)
    .sort_values("udział wśród wybranych", ascending=False)
)
organ_tab["średnie PQ w rundzie 2"] = (
    sel[sel["n_labeled"] == 2].groupby("organ")["pq_oracle"].mean()
)
organ_tab.round(3)

In [ ]:
freq = chosen.groupby("image_id")["seed"].nunique().sort_values(ascending=False)
hard = pd.DataFrame(
    {
        "wybrany w ilu seedach (z 6)": freq,
        "narząd": organs.reindex(freq.index),
        "średnie PQ w rundzie 2": sel[sel["n_labeled"] == 2]
        .groupby("image_id")["pq_oracle"]
        .mean()
        .reindex(freq.index),
        "jądra wyroczni": sel.groupby("image_id")["n_oracle"].first().reindex(freq.index),
    }
)
print(f"różnych obrazów wybranych w rundach 2-10: {len(freq)} z {len(POOL)} w puli")
hard.head(15).round(3)

## Koszt adnotacji w jądrach

Obraz to nie jest stała ilość pracy: liczba jąder (według wyroczni) różni się między obrazami kilkukrotnie. Jeśli
podgląd wybiera obrazy z większą liczbą jąder, to przy tej samej liczbie obrazów student dostaje więcej etykiet.

In [ ]:
nuclei = sel.groupby("image_id")[
    "n_oracle"
].first()  # każdy obraz puli jest nieoznaczony w rundzie 2 jakiegoś seeda


def labeled_nuclei(df: pd.DataFrame) -> pd.Series:
    total = [sum(nuclei[i] for i in json.loads(ids)) for ids in df["labeled_ids"]]
    return df.assign(nuclei=total).groupby("n_labeled")["nuclei"].mean()


cost = pd.DataFrame({"losowy": labeled_nuclei(rand), "podgląd": labeled_nuclei(peek)})
cost["podgląd / losowy"] = cost["podgląd"] / cost["losowy"]
print(
    f"obrazów puli z policzonymi jądrami: {len(nuclei)}; średnio {nuclei.mean():.0f} jąder na obraz (min {nuclei.min()}, max {nuclei.max()})"
)
cost.round(2)

## Czy trudność obrazu jest stała

Jeśli te same obrazy są trudne niezależnie od seeda i rundy, to „trudność” jest cechą obrazu (narząd, barwienie), a nie
stanu studenta, i selektor oparty na niepewności będzie w każdym seedzie wybierał podobny zestaw.

In [ ]:
r2 = sel[sel["n_labeled"] == 2].pivot(index="image_id", columns="seed", values="pq_oracle")
rho = r2.corr(method="spearman")
pairs = rho.where(np.triu(np.ones(rho.shape, dtype=bool), k=1)).stack()
print(
    f"runda 2, ranking PQ obrazów puli między seedami: średnia korelacja Spearmana = {pairs.mean():.2f} (min {pairs.min():.2f}, max {pairs.max():.2f})"
)
by_round = sel.pivot_table(
    index="image_id", columns="n_labeled", values="pq_oracle", aggfunc="mean"
)
print(
    "korelacja średniego PQ obrazu między rundą 2 a kolejnymi:",
    {
        int(n): round(by_round[2].corr(by_round[n], method="spearman"), 2)
        for n in by_round.columns
        if n != 2
    },
)